# Integrated EDK II Issue Triage Framework
## Tasks: Invalid Detection → Duplicate Detection → Priority Classification → Bug Assignment

> **Scope: 37 labeled issues** (consistent across all tasks — issues with known `First Assignee`)

**Best models (active by default):**
- **Invalid Detection** : `claude-sonnet-4-6` System A (catches invalids)
- **Duplicate Detection**: `claude-sonnet-4-6` System A (perfect score on 2-GT; zero FP)
- **Prioritization**    : `claude-sonnet-4-6` System A
- **Bug Assignment**    : `gpt-5.5` System A
Run cells top to bottom. Each task saves results to Google Drive under `DRIVE_DIR/results/`.


In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
!pip install openai anthropic chromadb sentence-transformers \
             rank_bm25 pandas scikit-learn tqdm -q


In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_DIR   = '/content/drive/MyDrive/triage_framework'
RESULTS_DIR = f'{DRIVE_DIR}/results'
CHROMA_DIR  = f'{DRIVE_DIR}/chroma_db'
# Try both folder names — Drive may add ' (1)' if folder already exists
_data_candidates = [f'{DRIVE_DIR}/data', f'{DRIVE_DIR} (1)/data']
DATA_DIR = next((p for p in _data_candidates if os.path.exists(p)), f'{DRIVE_DIR}/data')
print(f'DATA_DIR: {DATA_DIR}  (exists: {os.path.exists(DATA_DIR)})')

for d in [RESULTS_DIR, CHROMA_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Drive mounted. Results → {RESULTS_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR: /content/drive/MyDrive/triage_framework/data  (exists: True)
Drive mounted. Results → /content/drive/MyDrive/triage_framework/results


In [ ]:
# ============================================================
# CELL 3 — Config & API keys
# ============================================================
import os
from google.colab import userdata

# ── OpenAI ───────────────────────────────────────────────────
try:
    OPENAI_API_KEY    = userdata.get('OPENAI_API_KEY')
    OPENAI_ORG_ID     = userdata.get('OPENAI_ORG_ID')     or ''
    OPENAI_PROJECT_ID = userdata.get('OPENAI_PROJECT_ID') or ''
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
except:
    OPENAI_API_KEY    = input('OpenAI API key: ').strip()
    OPENAI_ORG_ID     = ''
    OPENAI_PROJECT_ID = ''

# ── Anthropic ────────────────────────────────────────────────
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY') or ''
except:
    ANTHROPIC_API_KEY = ''

print(f"OpenAI key    : {'set' if OPENAI_API_KEY    else 'NOT SET'}")
print(f"OpenAI org    : {'set' if OPENAI_ORG_ID     else 'not set'}")
print(f"OpenAI project: {'set' if OPENAI_PROJECT_ID else 'not set'}")
print(f"Anthropic key : {'set' if ANTHROPIC_API_KEY  else 'not set'}")

# ── Data path ─────────────────────────────────────────────────
# ── CSV file path (try multiple locations) ───────────────────
_csv_candidates = [
    '/content/drive/MyDrive/full-edkII-dd.csv', # Drive root (primary location)
    f'{DATA_DIR}/full-edkII-dd.csv',             # Drive/data folder
    f'{DRIVE_DIR}/full-edkII-dd.csv',            # Drive triage folder
    '/content/full-edkII-dd.csv',               # Colab root (manual upload)
]
GITHUB_CSV = next((p for p in _csv_candidates if os.path.exists(p)), None)

if GITHUB_CSV is None:
    print("\n  CSV NOT FOUND. Please do one of the following:")
    print("   Option 1: Upload 'full-edkII-dd.csv' using the Colab file panel (left sidebar)")
    print("             then run this cell again.")
    print("   Option 2: Copy the file to your Drive at:")
    print(f"             {DATA_DIR}/full-edkII-dd.csv")
    print("   Option 3: Set the path manually below:")
    GITHUB_CSV = '/content/full-edkII-dd.csv'  # ← change this path if needed
    raise FileNotFoundError(f"CSV not found. Set GITHUB_CSV manually or upload the file.")
else:
    print(f'\n✅ CSV found: {GITHUB_CSV}')

XML_FILES = [
    f'{DATA_DIR}/tianocore_closed_0_1000.xml',
    f'{DATA_DIR}/tianocore_closed_1000_2000.xml',
    f'{DATA_DIR}/tianocore_closed_2000_3000.xml',
    f'{DATA_DIR}/tianocore_closed_3000_4000.xml',
    f'{DATA_DIR}/tianocore_closed_4000_5000.xml',
]

# ── RAG flag ─────────────────────────────────────────────────
# Set to True ONLY if you want to use System B (RAG) for any task.
# If all tasks use System A (no RAG), keep this False to skip
# XML ingestion and Bugzilla indexing — saves significant setup time.
USE_RAG = False   # ← change to True if switching any task to System B

# ── Models ────────────────────────────────────────────────────
# Best models are ACTIVE (uncommented).
# Switch by commenting/uncommenting below.

# ── BEST MODELS (one per task, set individually in each task cell) ─────────
# Invalid Detection  → claude-sonnet-4-6 (System A)
# Duplicate Detect.  → claude-sonnet-4-6 (System A)
# Prioritization     → claude-sonnet-4-6 (System A)
# Bug Assignment     → gpt-5.5           (System A)

# ── All available models (for reference / batch runs) ────────
OPENAI_MODELS = [
    # 'gpt-4o-mini',
    # 'gpt-4o',
    # 'gpt-4.1-mini',
    # 'gpt-4.1',
    # 'gpt-5.4-mini',
    # 'gpt-5.4',
    'gpt-5.5',       # ← best for bug assignment
]

CLAUDE_MODELS = [
    # 'claude-haiku-4-5',
    'claude-sonnet-4-6',   # ← best for invalid, duplicate, priority
    # 'claude-opus-4-7',
] if ANTHROPIC_API_KEY else []

ALL_MODELS = OPENAI_MODELS + CLAUDE_MODELS

_RESPONSES_API_MODELS = {
    'gpt-5', 'gpt-5-mini', 'gpt-5.1', 'gpt-5.4', 'gpt-5.4-mini',
    'gpt-5.4-nano', 'gpt-5.4-pro', 'gpt-5.5', 'gpt-5.5-pro',
    'o3', 'o3-mini', 'o4-mini',
}

print(f'\nActive models ({len(ALL_MODELS)}): {ALL_MODELS}')


OpenAI key    : set
OpenAI org    : set
OpenAI project: set
Anthropic key : set

✅ CSV found: /content/drive/MyDrive/full-edkII-dd.csv

Active models (2): ['gpt-5.5', 'claude-sonnet-4-6']


In [ ]:
# ============================================================
# CELL 4 — Load & prepare data
# ============================================================
import pandas as pd
import re

import time
pipeline_start_time = time.time()  # total pipeline (setup + inference)
setup_start_time     = time.time()  # setup only
print(f'Pipeline started at: {time.strftime("%Y-%m-%d %H:%M:%S")} (from CSV load)')

df_all = pd.read_csv(GITHUB_CSV)

# Filter to 75 genuine GitHub-native issues
df_github = df_all[~df_all['Title'].str.contains(
    r'bugzilla bug \d+', case=False, na=False
)].copy()
print(f'GitHub-native issues : {len(df_github)}')

# Split labeled (37 with First Assignee) and unlabeled (38)
df_labeled   = df_github[df_github['First Assignee'].notna()].copy()
df_unlabeled = df_github[df_github['First Assignee'].isna()].copy()
print(f'Labeled (37 issues)  : {len(df_labeled)}')
print(f'Unlabeled (38)       : {len(df_unlabeled)}')

# ── Build issue dicts ─────────────────────────────────────────
def _clean(text):
    text = re.sub(r'https?://\S+', '<URL>', text or '')
    return re.sub(r'\s+', ' ', text).strip()

def build_issue(row):
    return {
        'id'            : str(int(row['Issue Number'])),
        'title'         : str(row['Title'] or ''),
        'body'          : _clean(str(row.get('body') or '')),
        'package'       : str(row.get('package') or '-'),
        'type'          : str(row.get('type')     or '-'),
        'priority'      : str(row.get('priority') or '-'),
        'comments'      : int(float(str(row.get('Comments', 0) or 0))),
        'state'         : str(row.get('state')    or ''),
        'true_assignee' : str(row['First Assignee']) if pd.notna(row.get('First Assignee')) else None,
        'text'          : _clean(f"Title: {row['Title']}\n\nDescription: {str(row.get('body',''))}"),
    }

labeled_issues   = [build_issue(r) for _, r in df_labeled.iterrows()]
unlabeled_issues = [build_issue(r) for _, r in df_unlabeled.iterrows()]
all_issues       = labeled_issues  # ← 37 labeled issues only (consistent across all tasks)
# all_issues     = labeled_issues + unlabeled_issues  # ← uncomment to run on all 75

KNOWN_ASSIGNEES = sorted(df_labeled['First Assignee'].dropna().unique().tolist())
print(f'\nKnown assignees ({len(KNOWN_ASSIGNEES)}): {KNOWN_ASSIGNEES}')

# ── Ground truth for 37 labeled issues ───────────────────────
# NOTE: None of the confirmed invalid or duplicate issues (#10579, #11258,
# #11453, #11480, #11948, #11462) appear in the 37 labeled issues.
# → GT for both tasks = empty set (zero-GT assumption).
# → Any flag by the model = false positive.
# → Best model = one that flags nothing (perfect precision, acc=1.0).

# Invalid GT — none of the 4 known invalids are in the 37 labeled issues
GT_INVALID_FULL = {'10579', '11258', '11453', '11480'}  # confirmed invalids (not in 37)
ACTIVE_GT_INVALID = set()   # ← empty for 37-issue evaluation

# Duplicate GT — confirmed pairs: {#11791↔#11790} and {#11948↔#11462}
# #11791 is a duplicate of #11790 (unidirectional)
# #11948 and #11462 are duplicates of each other (bidirectional)
# None of these 4 IDs appear in the 37 labeled issues → zero-GT assumption
GT_PAIRS_FULL         = [{'11791','11790'}, {'11948','11462'}]  # 2 confirmed pairs
GT_DUPLICATE_IDS_FULL = {'11791', '11790', '11948', '11462'}   # all 4 GT IDs (not in 37)
GT_DUPLICATE_IDS      = set()   # ← empty for 37-issue evaluation (zero-GT)

print(f'\nInvalid GT  (37-issue eval): {ACTIVE_GT_INVALID} — zero-GT assumption')
print(f'Duplicate GT (37-issue eval): {GT_DUPLICATE_IDS} — zero-GT assumption')
print(f'(Confirmed invalids {GT_INVALID_FULL} and duplicates {GT_DUPLICATE_IDS_FULL}')
print(f' exist in the unlabeled 38 issues, not in the labeled 37)')


Pipeline started at: 2026-05-25 23:53:48 (from CSV load)
GitHub-native issues : 75
Labeled (37 issues)  : 37
Unlabeled (38)       : 38

Known assignees (29): ['AndreasBBS', 'AshrafAliS', 'BritChesley', 'Damien-Chen', 'EricGao2015', 'HunterChang030', 'NingFengGit', 'SaloniKasbekar', 'Vogtinator', 'YangGangUEFI', 'ardbiesheuvel', 'bexcran', 'deeglaze', 'jxu023', 'lgao4', 'liyi77', 'makubacki', 'mdkinney', 'mikebeaton', 'os-d', 'philnoh2', 'praravin', 'praveensankarn', 'r1chard-lyu', 'vit9696', 'wenbhou', 'yechao-w', 'zevorn', 'zhurui22']

Invalid GT  (37-issue eval): set() — zero-GT assumption
Duplicate GT (37-issue eval): set() — zero-GT assumption
(Confirmed invalids {'11453', '11258', '10579', '11480'} and duplicates {'11948', '11462', '11790', '11791'}
 exist in the unlabeled 38 issues, not in the labeled 37)


In [ ]:
# ============================================================
# CELL 5 — Ingest XML corpus (only needed for System B / RAG)
# Skipped automatically if USE_RAG = False
# ============================================================
if not USE_RAG:
    print("⏭️  Skipping XML ingestion (USE_RAG=False — System A active)")
    xml_bugs             = []
    xml_invalid_examples = []
    xml_valid_examples   = []
    xml_dup_pairs        = {}
else:
    # ============================================================
    # CELL 5 — Ingest XML corpus
    # ============================================================
    import xml.etree.ElementTree as ET

    def _xml_body(bug_el):
        bodies = []
        for ld in bug_el.findall('long_desc'):
            t = ld.find('thetext')
            if t is not None and t.text:
                bodies.append(t.text.strip())
            if len(bodies) >= 3:
                break
        return '\n\n'.join(bodies)

    def load_xml_corpus(xml_paths):
        bugs = []
        for path in xml_paths:
            if not os.path.exists(path):
                print(f'  [skip] {path} not found')
                continue
            tree = ET.parse(path)
            root = tree.getroot()
            for bug_el in root.findall('bug'):
                def t(tag):
                    n = bug_el.find(tag)
                    return (n.text or '').strip() if n is not None else ''
                title      = t('short_desc')
                body       = _xml_body(bug_el)
                resolution = t('resolution')
                dup_node   = bug_el.find('dup_id')
                dup_id     = dup_node.text.strip() if dup_node is not None and dup_node.text else None
                bugs.append(dict(
                    id         = t('bug_id'),
                    title      = title,
                    body       = body,
                    package    = t('cf_package'),
                    component  = t('component'),
                    priority   = t('priority'),
                    resolution = resolution,
                    dup_id     = dup_id,
                    text       = _clean(f'Title: {title}\n\nDescription: {body}'),
                ))
        print(f'[ingest] XML corpus: {len(bugs):,} bugs from {len(xml_paths)} files')
        return bugs

    xml_bugs = load_xml_corpus(XML_FILES)

    # Separate invalid/valid XML examples for invalid detection
    xml_invalid_examples = [b for b in xml_bugs if b.get('resolution','').upper() == 'INVALID']
    xml_valid_examples   = [b for b in xml_bugs if b.get('resolution','').upper() not in ('INVALID','')]
    print(f'XML invalid examples : {len(xml_invalid_examples)}')
    print(f'XML valid examples   : {len(xml_valid_examples)}')

    # XML duplicate pairs
    xml_dup_pairs = {b['id']: b['dup_id'] for b in xml_bugs if b.get('dup_id')}
    print(f'XML duplicate pairs  : {len(xml_dup_pairs)}')



⏭️  Skipping XML ingestion (USE_RAG=False — System A active)


In [ ]:
# ============================================================
# CELL 6 — Build retrievers (ChromaDB + BM25)
# Run once — indexes persist to Drive
# ============================================================
import torch
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm

EMBED_MODEL = 'BAAI/bge-base-en-v1.5'
EMBED_BATCH = 64
RRF_K       = 60

def get_device():
    if torch.cuda.is_available():   return 'cuda'
    if hasattr(torch.backends,'mps') and torch.backends.mps.is_available(): return 'mps'
    return 'cpu'

def tokenise(text):
    text = re.sub(r'[^a-zA-Z0-9_]', ' ', (text or '').lower())
    return [t for t in text.split() if len(t) > 1]

def rrf(ranked_a, ranked_b, k=RRF_K):
    scores = {}
    for rank, doc_id in enumerate(ranked_a, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    for rank, doc_id in enumerate(ranked_b, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.__getitem__, reverse=True)

class BGEEmbeddingFunction(EmbeddingFunction):
    def __init__(self):
        device = get_device()
        print(f'[retriever] Loading {EMBED_MODEL} on {device}...')
        self._model = SentenceTransformer(EMBED_MODEL, device=device)
        print('[retriever] Model loaded')
    def __call__(self, input: Documents) -> Embeddings:
        return self._model.encode(
            list(input), batch_size=EMBED_BATCH,
            normalize_embeddings=True, show_progress_bar=False,
        ).tolist()

_embed_fn = BGEEmbeddingFunction()
_chroma   = chromadb.PersistentClient(path=CHROMA_DIR)

def _get_or_create(name):
    return _chroma.get_or_create_collection(
        name, embedding_function=_embed_fn,
        metadata={'hnsw:space': 'cosine'},
    )

def _index_docs(col, docs, label, force=False):
    if col.count() > 0 and not force:
        print(f'[retriever] {label}: {col.count():,} docs already indexed')
        return
    print(f'[retriever] Indexing {len(docs):,} {label} docs...')
    for i in tqdm(range(0, len(docs), EMBED_BATCH), desc=label):
        batch = docs[i:i+EMBED_BATCH]
        col.add(
            ids       = [d['id'] for d in batch],
            documents = [d['text'] for d in batch],
            metadatas = [{'title': d['title'][:500],
                          'package': str(d.get('package','') or ''),
                          'priority': str(d.get('priority','') or ''),
                          'body': str(d.get('body','') or '')[:300]} for d in batch],
        )
    print(f'[retriever] {label}: done ({col.count():,} docs)')

def _hybrid_search(query_text, col, bm25, bm25_ids, index, exclude_id, k):
    if col.count() == 0:
        return []
    fetch     = min(k * 4, col.count())
    dense_res = col.query(query_texts=[query_text], n_results=fetch, include=['distances'])
    dense_ids = [i for i in dense_res['ids'][0] if i != exclude_id]
    if bm25 is None or not bm25_ids:
        return [index[i] for i in dense_ids[:k] if i in index]
    scores    = bm25.get_scores(tokenise(query_text))
    sparse    = [bm25_ids[i]
                 for i in sorted(range(len(scores)), key=lambda x:-scores[x])
                 if bm25_ids[i] != exclude_id][:fetch]
    merged = rrf(dense_ids, sparse)[:k]
    return [index[i] for i in merged if i in index]

# ── XML collection (for duplicate + invalid RAG) ──────────────
if USE_RAG:
    _xml_col    = _get_or_create('xml_corpus_integrated')
    _index_docs(_xml_col, xml_bugs, 'XML corpus')
    _xml_by_id  = {b['id']: b for b in xml_bugs}
    _xml_bm25   = BM25Okapi([tokenise(b['text']) for b in xml_bugs]) if xml_bugs else None
    _xml_ids    = [b['id'] for b in xml_bugs]
else:
    _xml_col   = None
    _xml_by_id = {}
    _xml_bm25  = None
    _xml_ids   = []
    print('⏭️  XML collection skipped (USE_RAG=False)')

# ── GitHub issues collection (for duplicate candidate search) ─
_gh_col     = _get_or_create('github_issues_integrated')
_index_docs(_gh_col, all_issues, 'GitHub issues')
_gh_by_id   = {i['id']: i for i in all_issues}
_gh_bm25    = BM25Okapi([tokenise(i['text']) for i in all_issues]) if all_issues else None
_gh_ids     = [i['id'] for i in all_issues]

# ── Labeled issues collection (for bug assignment RAG) ────────
_lab_col    = _get_or_create('labeled_issues_integrated')
_index_docs(_lab_col, labeled_issues, 'Labeled issues')
_lab_by_id  = {i['id']: i for i in labeled_issues}
_lab_bm25   = BM25Okapi([tokenise(i['text']) for i in labeled_issues]) if labeled_issues else None
_lab_ids    = [i['id'] for i in labeled_issues]

# ── Bugzilla collection (for priority RAG) ────────────────────
if USE_RAG:
    _bz_col   = _get_or_create('bugzilla_priority_integrated')
    _bz_docs  = [{
        'id': b['id'], 'title': b['title'], 'body': b.get('body',''),
        'package': b.get('package',''), 'priority': b.get('priority',''),
        'text': _clean(f"Title: {b['title']}\n\nDescription: {b.get('body','')}")
    } for b in xml_bugs if b.get('priority','').lower() in ('low','medium','high')]
    _index_docs(_bz_col, _bz_docs, 'Bugzilla priority')
    _bz_bm25  = BM25Okapi([tokenise(d['text']) for d in _bz_docs]) if _bz_docs else None
    _bz_ids   = [d['id'] for d in _bz_docs]
    _bz_by_id = {d['id']: d for d in _bz_docs}
else:
    _bz_col   = None
    _bz_docs  = []
    _bz_bm25  = None
    _bz_ids   = []
    _bz_by_id = {}
    print('⏭️  Bugzilla collection skipped (USE_RAG=False)')

# ── Helper retrieval functions ─────────────────────────────────
def get_xml_context(issue, k=3):
    return _hybrid_search(issue['text'], _xml_col, _xml_bm25, _xml_ids, _xml_by_id, None, k)

def get_xml_dup_pairs(issue, k=3):
    similar = get_xml_context(issue, k=k)
    pairs = []
    for bug in similar:
        dup_id = bug.get('dup_id')
        pairs.append((bug, _xml_by_id.get(dup_id) if dup_id else None))
    return pairs

def get_dup_candidates(issue, k=5):
    return _hybrid_search(issue['text'], _gh_col, _gh_bm25, _gh_ids, _gh_by_id, issue['id'], k)

def get_similar_labeled(issue, k=5):
    return _hybrid_search(issue['text'], _lab_col, _lab_bm25, _lab_ids, _lab_by_id, issue['id'], k)

def get_bz_priority_context(issue, k=5):
    return _hybrid_search(issue['text'], _bz_col, _bz_bm25, _bz_ids, _bz_by_id, None, k)

print('\nAll retrievers ready.')


# ── Setup time ───────────────────────────────────────────────
setup_elapsed = time.time() - setup_start_time
inference_start_time = time.time()  # inference starts after retrievers are built
print(f'\n[SETUP] Runtime: {setup_elapsed:.1f}s ({setup_elapsed/60:.2f} min)')
print(f'[SETUP] Includes: CSV load, XML ingestion, ChromaDB + BM25 index build')


[retriever] Loading BAAI/bge-base-en-v1.5 on cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[retriever] Model loaded
⏭️  XML collection skipped (USE_RAG=False)
[retriever] GitHub issues: 37 docs already indexed
[retriever] Labeled issues: 37 docs already indexed
⏭️  Bugzilla collection skipped (USE_RAG=False)

All retrievers ready.

[SETUP] Runtime: 8.7s (0.14 min)
[SETUP] Includes: CSV load, XML ingestion, ChromaDB + BM25 index build


In [ ]:
# ============================================================
# CELL 7 — API clients & unified LLM caller
# ============================================================
import json, time, re
from openai import OpenAI

try:
    import anthropic
except ImportError:
    !pip install -q anthropic
    import anthropic

_openai_headers = {}
if OPENAI_ORG_ID:     _openai_headers['OpenAI-Organization'] = OPENAI_ORG_ID
if OPENAI_PROJECT_ID: _openai_headers['OpenAI-Project']      = OPENAI_PROJECT_ID

openai_client    = OpenAI(api_key=OPENAI_API_KEY, default_headers=_openai_headers)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print(f"OpenAI client   : set")
print(f"Anthropic client: {'set' if anthropic_client else 'not set (no key)'}")

def _parse_json(raw):
    """Extract JSON from raw LLM output — handles preamble, fences, bad escapes."""
    raw = re.sub(r'```(?:json)?\s*', '', raw)
    raw = re.sub(r'```', '', raw).strip()
    candidates = []
    depth, start = 0, None
    for idx, ch in enumerate(raw):
        if ch == '{':
            if depth == 0: start = idx
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start is not None:
                candidates.append(raw[start:idx+1]); start = None
    for blob in reversed(candidates):
        blob = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', blob)
        try:    return json.loads(blob)
        except: continue
    raise ValueError(f'Could not parse JSON from: {raw[:200]}')

def call_llm(system_prompt, user_content, model, max_tokens=512):
    """Route to OpenAI Chat / Responses API / Anthropic based on model name."""
    is_claude = model.startswith('claude')
    use_resp  = any(model.startswith(m) for m in _RESPONSES_API_MODELS)

    for attempt in range(3):
        try:
            if is_claude:
                resp = anthropic_client.messages.create(
                    model=model, max_tokens=max_tokens,
                    system=system_prompt,
                    messages=[{'role': 'user', 'content': user_content}],
                )
                raw = resp.content[0].text
            elif use_resp:
                full_input = system_prompt + '\n\n' + user_content
                resp = openai_client.responses.create(
                    model=model, input=full_input,
                    extra_headers=_openai_headers,
                )
                raw = resp.output_text
            else:
                resp = openai_client.chat.completions.create(
                    model=model, temperature=0.0, max_tokens=max_tokens,
                    messages=[
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user',   'content': user_content},
                    ],
                    extra_headers=_openai_headers,
                )
                raw = resp.choices[0].message.content
            return _parse_json(raw)
        except Exception as e:
            print(f'  [attempt {attempt+1}] {type(e).__name__}: {e}')
            if attempt < 2: time.sleep(2 ** attempt)
    return None

print('call_llm ready.')


OpenAI client   : set
Anthropic client: set
call_llm ready.


## Task 1 — Invalid Issue Detection

Best model: `claude-sonnet-4-6` System A  
Ground truth: 4 confirmed invalids (#10579, #11258, #11453, #11480)

In [ ]:
invalid_start = time.time()
print(f'[INVALID] Starting...')

# ============================================================
# CELL 8 — Invalid Issue Detection
# Best model: claude-sonnet-4-6 System A
# ============================================================
import csv, random
from tqdm.notebook import tqdm

random.seed(42)

# ── Active model for invalid detection ───────────────────────
INVALID_MODEL  = 'claude-sonnet-4-6'   # ← best model (active) — highest Mac-F1 on 4-GT
# INVALID_MODEL = 'gpt-4o-mini'
# INVALID_MODEL = 'gpt-4o'
# INVALID_MODEL = 'gpt-4.1-mini'
# INVALID_MODEL = 'gpt-4.1'
# INVALID_MODEL = 'gpt-5.4-mini'
# INVALID_MODEL = 'gpt-5.4'        # ← best for 1-GT setup (System B)
# INVALID_MODEL = 'gpt-5.5'
# INVALID_MODEL = 'claude-haiku-4-5'
# INVALID_MODEL = 'claude-opus-4-7'

INVALID_SYSTEM = 'A'   # ← best system (active)
# INVALID_SYSTEM = 'B'
XML_CONTEXT_K  = 3

# ── Mask state to prevent label leakage ──────────────────────
def mask_state(state):
    return '' if str(state).strip().lower() == 'invalid' else state

invalid_test_issues = []
for issue in labeled_issues:  # 37 labeled issues
    state_orig   = issue['state']
    state_masked = mask_state(state_orig)
    invalid_test_issues.append({
        **issue,
        'state'       : state_masked,
        'true_invalid': issue['id'] in ACTIVE_GT_INVALID,
    })

# ── Few-shot examples ─────────────────────────────────────────
fewshot_invalid = random.sample(xml_invalid_examples, min(4, len(xml_invalid_examples)))
fewshot_valid   = random.sample(xml_valid_examples,   min(4, len(xml_valid_examples)))

def build_invalid_fewshot():
    block = "EXAMPLES FROM THIS PROJECT'S HISTORY:\n"
    for b in fewshot_invalid:
        block += f"\nTitle      : {b['title']}\nPackage    : {b.get('package','-')}\nResolution : INVALID\n"
    for b in fewshot_valid:
        block += f"\nTitle      : {b['title']}\nPackage    : {b.get('package','-')}\nResolution : VALID\n"
    return block

_INVALID_SYS = """You are an experienced EDK II firmware project maintainer.

Decide whether a GitHub issue is VALID or INVALID for the EDK II project tracker.

An issue is INVALID if:
- It is spam or completely unrelated to firmware development
  (e.g. printer support, social media, password reset, unrelated software)
- It is a general usage question, not a bug report or feature request
- It describes something that works as designed — the reporter misunderstood the behavior
- It does not contain enough information to reproduce or investigate

An issue is VALID if:
- It is a genuine bug report with a reproducible problem
- It is a legitimate feature request for EDK II
- It describes a real security vulnerability or build failure
- It has enough information to investigate
- It relates to EDK II build infrastructure such as build tools, compilers, or linkers
- It relates to EDK II coding style enforcement tools
- It involves any EDK II package, driver, protocol, or firmware component
- When in doubt, treat the issue as VALID

{fewshot}

Now classify the following issue.
Respond with ONLY valid JSON:
{{
  "is_invalid": <bool>,
  "confidence": <float 0.0-1.0>,
  "reason"    : "<one sentence>"
}}"""

_INVALID_SYS_B = _INVALID_SYS + """

Use the following similar historical bugs as additional context:
{xml_context}"""

def fmt_issue_invalid(issue):
    return (f"Title   : {issue['title']}\n"
            f"Package : {issue.get('package','-')}\n"
            f"Type    : {issue.get('type','-')}\n"
            f"Comments: {issue.get('comments',0)}\n"
            f"State   : {issue.get('state','')}")

def predict_invalid(issue, model=INVALID_MODEL, system=INVALID_SYSTEM):
    fewshot = build_invalid_fewshot()
    if system == 'A':
        sys_prompt = _INVALID_SYS.format(fewshot=fewshot)
    else:
        similar   = get_xml_context(issue, k=XML_CONTEXT_K)
        xml_block = '\n'.join(
            f"- {b['title']} [{b.get('package','-')}] → {b.get('resolution','')}"
            for b in similar
        )
        sys_prompt = _INVALID_SYS_B.format(fewshot=fewshot, xml_context=xml_block)
    user = "ISSUE TO CLASSIFY:\n" + fmt_issue_invalid(issue)
    result = call_llm(sys_prompt, user, model, max_tokens=200)
    if result is None:
        return {'is_invalid': False, 'confidence': 0.0, 'reason': 'error'}
    result['is_invalid'] = bool(result.get('is_invalid', False))
    return result

# ── Run invalid detection ─────────────────────────────────────
print(f"\nRunning invalid detection — model: {INVALID_MODEL}, system: {INVALID_SYSTEM}")
print(f"Ground truth invalids: {ACTIVE_GT_INVALID}  ({len(ACTIVE_GT_INVALID)} issues)")

invalid_results = []
path_inv_csv = f'{RESULTS_DIR}/invalid_{INVALID_SYSTEM}_{INVALID_MODEL.replace(".","_")}.csv'

for issue in tqdm(invalid_test_issues, desc=f'Invalid | {INVALID_MODEL}'):
    try:
        pred = predict_invalid(issue)
        pred.update({
            'model'       : INVALID_MODEL,
            'system'      : INVALID_SYSTEM,
            'issue_id'    : issue['id'],
            'title'       : issue['title'],
            'package'     : issue['package'],
            'true_invalid': issue['true_invalid'],
        })
        invalid_results.append(pred)
    except Exception as e:
        print(f"  Skipped #{issue['id']}: {e}")

# Save CSV
keys = ['model','system','issue_id','title','package','is_invalid','confidence','reason','true_invalid']
with open(path_inv_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
    writer.writeheader(); writer.writerows(invalid_results)

n_flagged = sum(1 for r in invalid_results if r['is_invalid'])
print(f"\nFlagged as invalid: {n_flagged}/{len(invalid_results)}")
print(f"Saved → {path_inv_csv}")
invalid_elapsed = time.time() - invalid_start
print(f'[INVALID] Runtime: {invalid_elapsed:.1f}s ({invalid_elapsed/60:.2f} min)')

# ── Evaluation (zero-GT: all 37 are valid) ───────────────────
# GT = empty set → TP=0, FN=0 always
# FP = flagged issues (all wrong under this assumption)
# TN = not flagged (correct)
flagged_ids = {r['issue_id'] for r in invalid_results if r['is_invalid']}
n  = len(invalid_test_issues)
TP = 0
FP = len(flagged_ids)
TN = n - FP
FN = 0

prec_pos = 1.0 if FP == 0 else 0.0   # 1.0 = nothing flagged (perfect)
rec_pos  = 1.0                         # defined as 1.0 (FN=0)
f1_pos   = 2*prec_pos*rec_pos/(prec_pos+rec_pos) if (prec_pos+rec_pos)>0 else 0.0
prec_neg = TN/(TN+FN) if (TN+FN)>0 else 0.0
rec_neg  = TN/(TN+FP) if (TN+FP)>0 else 0.0
f1_neg   = 2*prec_neg*rec_neg/(prec_neg+rec_neg) if (prec_neg+rec_neg)>0 else 0.0
mac_f1   = (f1_pos + f1_neg) / 2
acc      = TN / n

print(f"\n{'='*60}")
print(f"  INVALID DETECTION RESULTS (zero-GT: all 37 assumed valid)")
print(f"{'='*60}")
print(f"  TP={TP}  FP={FP}  TN={TN}  FN={FN}")
print(f"  P+={prec_pos:.4f}  R+={rec_pos:.4f}  F1+={f1_pos:.4f}")
print(f"  P-={prec_neg:.4f}  R-={rec_neg:.4f}  F1-={f1_neg:.4f}")
print(f"  Mac-F1={mac_f1:.4f}  Acc={acc:.4f}")
print(f"  Note: P+=1.0 means model flagged nothing (conservative/best under zero-GT)")
print(f"{'='*60}")

# ── Filter valid issues for next tasks ────────────────────────
invalid_issue_ids     = {r['issue_id'] for r in invalid_results if r['is_invalid']}

# ── Save evaluation metrics to CSV ───────────────────────────
import csv as _csv
path_inv_metrics = f'{RESULTS_DIR}/metrics_invalid_{INVALID_SYSTEM}_{INVALID_MODEL.replace(".","_")}.csv'
with open(path_inv_metrics, 'w', newline='') as _f:
    _w = _csv.DictWriter(_f, fieldnames=['model','system','n_issues','TP','FP','TN','FN',
                                          'prec_pos','rec_pos','f1_pos',
                                          'prec_neg','rec_neg','f1_neg',
                                          'mac_p','mac_r','mac_f1',
                                          'wgt_p','wgt_r','wgt_f1','acc'])
    _w.writeheader()
    sup_neg_inv = n
    wgt_p_inv  = prec_neg
    wgt_r_inv  = rec_neg
    wgt_f1_inv = f1_neg
    _w.writerow(dict(model=INVALID_MODEL, system=INVALID_SYSTEM, n_issues=n,
                     TP=TP, FP=FP, TN=TN, FN=FN,
                     prec_pos=round(prec_pos,4), rec_pos=round(rec_pos,4), f1_pos=round(f1_pos,4),
                     prec_neg=round(prec_neg,4), rec_neg=round(rec_neg,4), f1_neg=round(f1_neg,4),
                     mac_p=round((prec_pos+prec_neg)/2,4), mac_r=round((rec_pos+rec_neg)/2,4),
                     mac_f1=round(mac_f1,4),
                     wgt_p=round(wgt_p_inv,4), wgt_r=round(wgt_r_inv,4), wgt_f1=round(wgt_f1_inv,4),
                     acc=round(acc,4)))
print(f"Metrics saved → {path_inv_metrics}")

print(f"\nIssues passing to next task: {len(all_issues)}")


[INVALID] Starting...

Running invalid detection — model: claude-sonnet-4-6, system: A
Ground truth invalids: set()  (0 issues)


Invalid | claude-sonnet-4-6:   0%|          | 0/37 [00:00<?, ?it/s]


Flagged as invalid: 1/37
Saved → /content/drive/MyDrive/triage_framework/results/invalid_A_claude-sonnet-4-6.csv
[INVALID] Runtime: 80.7s (1.35 min)

  INVALID DETECTION RESULTS (zero-GT: all 37 assumed valid)
  TP=0  FP=1  TN=36  FN=0
  P+=0.0000  R+=1.0000  F1+=0.0000
  P-=1.0000  R-=0.9730  F1-=0.9863
  Mac-F1=0.4932  Acc=0.9730
  Note: P+=1.0 means model flagged nothing (conservative/best under zero-GT)
Metrics saved → /content/drive/MyDrive/triage_framework/results/metrics_invalid_A_claude-sonnet-4-6.csv

Issues passing to next task: 37


## Task 2 — Duplicate Detection

Best model: `claude-sonnet-4-6` System A (best on both 0-pair and 2-pair GT, FP=0)

**Ground truth pairs:**
- **#11791** → duplicate of **#11790** (unidirectional)
- **#11948** ↔ **#11462** (bidirectional)

**Note:** None of these 4 IDs appear in the 37 labeled issues → zero-GT assumption applies for this evaluation set.


In [ ]:
dup_start = time.time()
print(f'[DUPLICATE] Starting...')

# ============================================================
# CELL 9 — Duplicate Detection
# Best model: claude-sonnet-4-6 System A
# ============================================================
from tqdm.notebook import tqdm
import csv

DUP_MODEL      = 'claude-sonnet-4-6'   # ← best model (active)
                                        #   0-pair GT: best on A & B (FP=0)
                                        #   2-pair GT: best on A & B (FP=0, Mac-F1=0.826)
# DUP_MODEL    = 'gpt-4o-mini'         # ← tied best on 2-pair GT System A
# DUP_MODEL    = 'gpt-4o'
# DUP_MODEL    = 'gpt-4.1-mini'
# DUP_MODEL    = 'gpt-4.1'
# DUP_MODEL    = 'gpt-5.4-mini'
# DUP_MODEL    = 'gpt-5.4'
# DUP_MODEL    = 'gpt-5.5'
# DUP_MODEL    = 'claude-haiku-4-5'
# DUP_MODEL    = 'claude-opus-4-7'     # ← tied best on 2-pair GT System A

DUP_SYSTEM     = 'A'   # ← best system (active)
# DUP_SYSTEM   = 'B'
DUP_CANDIDATE_K = 5
XML_DUP_K       = 3

_PROMPT_DUP_A = """Find duplicate GitHub issues.

Given a QUERY ISSUE and a list of CANDIDATE ISSUES, is the query a duplicate of any candidate?

Two issues are duplicates only if they report the exact same bug or request the exact same change.
Same package or similar topic is not enough.
When in doubt return is_duplicate: false.

Respond with ONLY valid JSON:
{{
  "is_duplicate": <bool>,
  "duplicate_of": <issue number as string, or null>,
  "confidence"  : <float 0.0-1.0>,
  "reason"      : "<one sentence>"
}}"""

_PROMPT_DUP_B = """Find duplicate GitHub issues.

You will be given DUPLICATE EXAMPLES from this codebase's history, a QUERY ISSUE, and CANDIDATE ISSUES.

The duplicate examples are real pairs of bugs that were marked as the same issue in this project.
Use them to understand what counts as a duplicate in this specific codebase.

Two GitHub issues are duplicates only if they report the exact same bug or request the exact same change.
Same package or similar topic is not enough.
When in doubt return is_duplicate: false.

Respond with ONLY valid JSON:
{{
  "is_duplicate": <bool>,
  "duplicate_of": <issue number as string, or null>,
  "confidence"  : <float 0.0-1.0>,
  "reason"      : "<one sentence>"
}}"""

def fmt_github_issue(issue):
    return (f"Issue #{issue['id']}\n"
            f"Title  : {issue['title']}\n"
            f"Package: {issue.get('package','-')}")

def fmt_xml_pair(bug, dup_bug):
    r = "--- Example duplicate pair ---\n"
    r += f"Bug A: {bug['title']}\n  {(bug.get('body') or '')[:200]}\n"
    if dup_bug:
        r += f"Bug B (same as A): {dup_bug['title']}\n  {(dup_bug.get('body') or '')[:200]}\n"
    return r

def predict_duplicate(issue, model=DUP_MODEL, system=DUP_SYSTEM):
    candidates = get_dup_candidates(issue, k=DUP_CANDIDATE_K)
    if system == 'A':
        sys_prompt = _PROMPT_DUP_A
        user = "QUERY ISSUE:\n" + fmt_github_issue(issue)
        user += "\n\nCANDIDATE ISSUES:\n"
        for i, c in enumerate(candidates, 1):
            user += f"\n[{i}]\n{fmt_github_issue(c)}\n"
    else:
        xml_pairs = get_xml_dup_pairs(issue, k=XML_DUP_K)
        sys_prompt = _PROMPT_DUP_B
        user = ""
        pairs_with_both = [(a,b) for a,b in xml_pairs if b is not None]
        if pairs_with_both:
            user += "DUPLICATE EXAMPLES FROM THIS CODEBASE:\n"
            for bug, dup_bug in pairs_with_both:
                user += f"\n{fmt_xml_pair(bug, dup_bug)}\n"
            user += "\n---\n\n"
        user += "QUERY ISSUE:\n" + fmt_github_issue(issue)
        user += "\n\nCANDIDATE ISSUES:\n"
        for i, c in enumerate(candidates, 1):
            user += f"\n[{i}]\n{fmt_github_issue(c)}\n"

    result = call_llm(sys_prompt, user, model, max_tokens=1500)
    if result is None:
        return {'is_duplicate': False, 'duplicate_of': None, 'confidence': 0.0}
    return result

# ── Run duplicate detection ───────────────────────────────────
issues_for_dup = labeled_issues if 'labeled_issues' in dir() else labeled_issues
issues_for_dup = [i for i in issues_for_dup
                  if not re.search(r'bugzilla bug \d+', i['title'], re.IGNORECASE)]

print(f"\nRunning duplicate detection — model: {DUP_MODEL}, system: {DUP_SYSTEM}")
print(f"Issues to process: {len(issues_for_dup)}")
print(f"GT pairs (full): {GT_PAIRS_FULL}")
print(f"GT IDs (full)  : {GT_DUPLICATE_IDS_FULL}")
print(f"GT for eval    : {GT_DUPLICATE_IDS} (zero-GT — none of the 4 IDs in labeled 37)")

dup_results  = []
path_dup_csv = f'{RESULTS_DIR}/dup_{DUP_SYSTEM}_{DUP_MODEL.replace(".","_")}.csv'

for issue in tqdm(issues_for_dup, desc=f'Dup | {DUP_MODEL}'):
    try:
        pred = predict_duplicate(issue)
        dup_title = _gh_by_id.get(str(pred.get('duplicate_of') or ''), {}).get('title', '')
        pred.update({
            'model'          : DUP_MODEL,
            'system'         : DUP_SYSTEM,
            'issue_id'       : issue['id'],
            'title'          : issue['title'],
            'package'        : issue['package'],
            'duplicate_title': dup_title,
        })
        dup_results.append(pred)
    except Exception as e:
        print(f"  Skipped #{issue['id']}: {e}")

# Save CSV
keys = ['model','system','issue_id','title','package','is_duplicate','duplicate_of','duplicate_title','confidence','reason']
with open(path_dup_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
    writer.writeheader(); writer.writerows(dup_results)

n_flagged = sum(1 for r in dup_results if r['is_duplicate'])
print(f"\nFlagged as duplicate: {n_flagged}/{len(dup_results)}")
print(f"Saved → {path_dup_csv}")
dup_elapsed = time.time() - dup_start
print(f'[DUPLICATE] Runtime: {dup_elapsed:.1f}s ({dup_elapsed/60:.2f} min)')

# ── Evaluation (zero-GT: all 37 are non-duplicates) ──────────
# GT = empty set → TP=0, FN=0 always
# FP = flagged issues (all wrong under this assumption)
# TN = not flagged (correct)
flagged_ids = {r['issue_id'] for r in dup_results if r['is_duplicate']}
n  = len(issues_for_dup)
TP = 0
FP = len(flagged_ids)
TN = n - FP
FN = 0

prec_pos = 1.0 if FP == 0 else 0.0   # 1.0 = nothing flagged (perfect)
rec_pos  = 1.0                         # defined as 1.0 (FN=0)
f1_pos   = 2*prec_pos*rec_pos/(prec_pos+rec_pos) if (prec_pos+rec_pos)>0 else 0.0
prec_neg = TN/(TN+FN) if (TN+FN)>0 else 0.0
rec_neg  = TN/(TN+FP) if (TN+FP)>0 else 0.0
f1_neg   = 2*prec_neg*rec_neg/(prec_neg+rec_neg) if (prec_neg+rec_neg)>0 else 0.0
mac_f1   = (f1_pos + f1_neg) / 2
acc      = TN / n

print(f"\n{'='*65}")
print(f"  DUPLICATE DETECTION RESULTS (zero-GT: all 37 assumed non-duplicate)")
print(f"{'='*65}")
print(f"  TP={TP}  FP={FP}  TN={TN}  FN={FN}")
print(f"  P+={prec_pos:.4f}  R+={rec_pos:.4f}  F1+={f1_pos:.4f}")
print(f"  P-={prec_neg:.4f}  R-={rec_neg:.4f}  F1-={f1_neg:.4f}")
print(f"  Mac-F1={mac_f1:.4f}  Acc={acc:.4f}")
print(f"  Note: P+=1.0 means model flagged nothing (conservative/best under zero-GT)")
print(f"{'='*65}")

# Filter duplicate issues from pipeline
duplicate_issue_ids    = {r['issue_id'] for r in dup_results if r['is_duplicate']}

# ── Save evaluation metrics to CSV ───────────────────────────
path_dup_metrics = f'{RESULTS_DIR}/metrics_dup_{DUP_SYSTEM}_{DUP_MODEL.replace(".","_")}.csv'
with open(path_dup_metrics, 'w', newline='') as _f:
    _w = csv.DictWriter(_f, fieldnames=['model','system','n_issues','TP','FP','TN','FN',
                                         'prec_pos','rec_pos','f1_pos',
                                         'prec_neg','rec_neg','f1_neg',
                                         'mac_p','mac_r','mac_f1',
                                         'wgt_p','wgt_r','wgt_f1','acc'])
    _w.writeheader()
    wgt_p_dup  = prec_neg
    wgt_r_dup  = rec_neg
    wgt_f1_dup = f1_neg
    _w.writerow(dict(model=DUP_MODEL, system=DUP_SYSTEM, n_issues=n,
                     TP=TP, FP=FP, TN=TN, FN=FN,
                     prec_pos=round(prec_pos,4), rec_pos=round(rec_pos,4), f1_pos=round(f1_pos,4),
                     prec_neg=round(prec_neg,4), rec_neg=round(rec_neg,4), f1_neg=round(f1_neg,4),
                     mac_p=round((prec_pos+prec_neg)/2,4), mac_r=round((rec_pos+rec_neg)/2,4),
                     mac_f1=round(mac_f1,4),
                     wgt_p=round(wgt_p_dup,4), wgt_r=round(wgt_r_dup,4), wgt_f1=round(wgt_f1_dup,4),
                     acc=round(acc,4)))
print(f"Metrics saved → {path_dup_metrics}")

print(f"\nNon-duplicate issues passing to next task: {len(issues_for_dup)}")


[DUPLICATE] Starting...

Running duplicate detection — model: claude-sonnet-4-6, system: A
Issues to process: 37
GT pairs (full): [{'11790', '11791'}, {'11462', '11948'}]
GT IDs (full)  : {'11948', '11462', '11790', '11791'}
GT for eval    : set() (zero-GT — none of the 4 IDs in labeled 37)


Dup | claude-sonnet-4-6:   0%|          | 0/37 [00:00<?, ?it/s]


Flagged as duplicate: 0/37
Saved → /content/drive/MyDrive/triage_framework/results/dup_A_claude-sonnet-4-6.csv
[DUPLICATE] Runtime: 92.0s (1.53 min)

  DUPLICATE DETECTION RESULTS (zero-GT: all 37 assumed non-duplicate)
  TP=0  FP=0  TN=37  FN=0
  P+=1.0000  R+=1.0000  F1+=1.0000
  P-=1.0000  R-=1.0000  F1-=1.0000
  Mac-F1=1.0000  Acc=1.0000
  Note: P+=1.0 means model flagged nothing (conservative/best under zero-GT)
Metrics saved → /content/drive/MyDrive/triage_framework/results/metrics_dup_A_claude-sonnet-4-6.csv

Non-duplicate issues passing to next task: 37


## Task 3 — Priority Classification

Best model: `claude-sonnet-4-6` System A

In [ ]:
# ============================================================
# CELL 10 — Priority Classification
# Best model: claude-sonnet-4-6 System A
# ============================================================
from tqdm.notebook import tqdm
import csv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score

pri_start = time.time()
print(f'[PRIORITY] Starting...')

PRI_MODEL  = 'claude-sonnet-4-6'   # ← best model (active) — Acc=0.627, Mac-F1=0.613
# PRI_MODEL = 'gpt-4o-mini'
# PRI_MODEL = 'gpt-4o'
# PRI_MODEL = 'gpt-4.1-mini'
# PRI_MODEL = 'gpt-4.1'
# PRI_MODEL = 'gpt-5.4-mini'
# PRI_MODEL = 'gpt-5.4'
# PRI_MODEL = 'gpt-5.5'
# PRI_MODEL = 'claude-haiku-4-5'
# PRI_MODEL = 'claude-opus-4-7'

PRI_SYSTEM = 'A'   # ← best system (active)
# PRI_SYSTEM = 'B'

RAG_TOP_K       = 5
PRIORITY_LABELS = ['low', 'medium', 'high']

DOMAIN_RULES = """\
You are a bug priority classifier for the tianocore/edk2 UEFI firmware repository.

PRIORITY DEFINITIONS:
- high   : Crashes, data corruption, security vulnerabilities that directly
           apply to the UEFI threat model, or build failures that block ALL
           configurations for ALL users.
- medium : Functional bugs with workarounds, build failures limited to a
           specific compiler or toolchain, missing features required for spec
           compliance, race conditions that may cause instability.
- low    : Cosmetic issues, code quality improvements, warnings that do not
           block builds, missing optional features, documentation changes,
           bugs in non-default configurations.

DOMAIN KNOWLEDGE RULES for edk2 priority triage:

1. CVE / security updates are NOT automatically high. UEFI firmware has a
   different threat model than an OS. If the vulnerable code path in a library
   (e.g. OpenSSL) is not reachable during UEFI execution, the patch priority
   reflects maintenance urgency, not security severity.

2. Build failures severity depends on scope. A failure that blocks ALL users
   regardless of compiler or configuration is high. A failure limited to one
   specific compiler (e.g. VS2022 only, CLANG only) or one toolchain variant
   (e.g. CLANGPDB, CLANGDWARF) is medium. A warning that does not block the
   build at all is low.

3. UI and display bugs are low unless they completely prevent the user from
   interacting with the firmware (e.g. no input accepted at all).

4. Missing feature additions or spec compliance gaps are medium if the spec
   is published and widely adopted, low if the feature is optional or cosmetic.

5. ALWAYS use the training examples as your primary signal. If a similar bug
   in the examples was labeled low, do not escalate without a concrete reason
   specific to this bug that is not present in the example.
"""

JSON_INSTRUCTION = """\
Before predicting, scan the examples above for the most similar issue — same
package, same bug type. Use that example's label as your anchor. Only deviate
if there is a concrete, specific reason this issue is clearly more or less
severe than the similar example.

Only choose from: low, medium, high

Respond ONLY with a JSON object, nothing else — no prose before or after:
{"predicted_priority": "<low|medium|high>", "confidence": <float 0.0-1.0>, "reason": "<one sentence>"}"""

def fmt_issue_priority(issue, include_priority=True):
    body = str(issue.get('body','') or '')[:300].replace('\n',' ')
    pkg  = issue.get('package','') or 'unknown'
    text = f"Title   : {issue['title']}\nPackage : {pkg}\nBody    : {body}"
    if include_priority:
        text += f"\nPriority: {issue.get('priority','-')}"
    return text

def build_priority_prompt_A(test_issue, all_issues_df):
    train    = [i for i in all_issues_df
                if i['id'] != test_issue['id'] and i.get('priority','').lower() in PRIORITY_LABELS]
    examples = '\n\n'.join(fmt_issue_priority(i) for i in train)
    test     = fmt_issue_priority(test_issue, include_priority=False)
    return (DOMAIN_RULES + '\n---\nPast GitHub issues and their priority labels:\n\n'
            + examples + '\n\n---\nNew issue to classify:\n\n' + test
            + '\n\n' + JSON_INSTRUCTION), 'Classify the issue above.'

def build_priority_prompt_B(test_issue, all_issues_df):
    train    = [i for i in all_issues_df
                if i['id'] != test_issue['id'] and i.get('priority','').lower() in PRIORITY_LABELS]
    examples = '\n\n'.join(fmt_issue_priority(i) for i in train)
    rag_bugs = get_bz_priority_context(test_issue, k=RAG_TOP_K)
    rag_ctx  = '\n\n'.join(
        f"Title   : {b['title']}\nPackage : {b.get('package','')}\n"
        f"Body    : {str(b.get('body',''))[:200]}\nPriority: {b.get('priority','')}"
        for b in rag_bugs
    )
    test = fmt_issue_priority(test_issue, include_priority=False)
    return (DOMAIN_RULES + '\n---\nSIMILAR HISTORICAL BUGS from Bugzilla (RAG context):\n\n'
            + rag_ctx + '\n\n---\nPast GitHub issues:\n\n' + examples
            + '\n\n---\nNew issue to classify:\n\n' + test
            + '\n\n' + JSON_INSTRUCTION), 'Classify the issue above.'

def predict_priority(issue, all_issues_list, model=PRI_MODEL, system=PRI_SYSTEM):
    if system == 'A':
        sys_p, user_p = build_priority_prompt_A(issue, all_issues_list)
    else:
        sys_p, user_p = build_priority_prompt_B(issue, all_issues_list)
    result = call_llm(sys_p, user_p if user_p and user_p.strip() else 'Classify the issue above.', model, max_tokens=200)
    if result is None:
        return {'predicted_priority': None, 'confidence': 0.0, 'reason': 'error'}
    pred = result.get('predicted_priority','').strip().lower()
    if pred not in PRIORITY_LABELS:
        result['predicted_priority'] = None
    return result

# ── Run priority classification ───────────────────────────────
issues_with_priority = [i for i in labeled_issues if i.get('priority','').lower() in PRIORITY_LABELS]
issues_for_pri = issues_with_priority  # always all labeled issues with known priority

print(f"\nRunning priority classification — model: {PRI_MODEL}, system: {PRI_SYSTEM}")
print(f"Issues to process: {len(issues_for_pri)}")

pri_results  = []
path_pri_csv = f'{RESULTS_DIR}/priority_{PRI_SYSTEM}_{PRI_MODEL.replace(".","_")}.csv'

for issue in tqdm(issues_for_pri, desc=f'Priority | {PRI_MODEL}'):
    try:
        pred = predict_priority(issue, issues_with_priority)
        pred.update({
            'model'         : PRI_MODEL,
            'system'        : PRI_SYSTEM,
            'issue_id'      : issue['id'],
            'title'         : issue['title'],
            'package'       : issue['package'],
            'true_priority' : issue.get('priority',''),
        })
        pri_results.append(pred)
    except Exception as e:
        print(f"  Skipped #{issue['id']}: {e}")

# Save predictions CSV
keys = ['model','system','issue_id','title','package','predicted_priority','true_priority','confidence','reason']
with open(path_pri_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(pri_results)
print(f"Saved → {path_pri_csv}")

pri_elapsed = time.time() - pri_start
print(f'[PRIORITY] Runtime: {pri_elapsed:.1f}s ({pri_elapsed/60:.2f} min)')

# ── Evaluation ────────────────────────────────────────────────
labeled_pri = [r for r in pri_results
               if r.get('true_priority','').lower() in PRIORITY_LABELS
               and r.get('predicted_priority') is not None]
if labeled_pri:
    y_true  = [r['true_priority']      for r in labeled_pri]
    y_pred  = [r['predicted_priority'] for r in labeled_pri]
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mac_p   = precision_score(y_true, y_pred, average='macro',    zero_division=0)
    mac_r   = recall_score(   y_true, y_pred, average='macro',    zero_division=0)
    mac_f1  = f1_score(       y_true, y_pred, average='macro',    zero_division=0)
    wgt_p   = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    wgt_r   = recall_score(   y_true, y_pred, average='weighted', zero_division=0)
    wgt_f1  = f1_score(       y_true, y_pred, average='weighted', zero_division=0)
    correct = sum(1 for a, b in zip(y_true, y_pred) if a == b)
    print(f"\n{'='*55}")
    print(f"  PRIORITY RESULTS ({len(labeled_pri)} evaluated)")
    print(f"{'='*55}")
    print(f"  Correct    : {correct}/{len(labeled_pri)}")
    print(f"  Accuracy   : {acc:.4f}  |  Bal-Acc: {bal_acc:.4f}")
    print(f"  Mac  P/R/F1: {mac_p:.3f} / {mac_r:.3f} / {mac_f1:.3f}")
    print(f"  Wgt  P/R/F1: {wgt_p:.3f} / {wgt_r:.3f} / {wgt_f1:.3f}")
    print(f"{'='*55}")

    # Save evaluation metrics CSV
    path_pri_metrics = f'{RESULTS_DIR}/metrics_priority_{PRI_SYSTEM}_{PRI_MODEL.replace(".","_")}.csv'
    with open(path_pri_metrics, 'w', newline='') as _f:
        _w = csv.DictWriter(_f, fieldnames=['model','system','n_issues','acc','bal_acc',
                                             'mac_p','mac_r','mac_f1',
                                             'wgt_p','wgt_r','wgt_f1'])
        _w.writeheader()
        _w.writerow(dict(model=PRI_MODEL, system=PRI_SYSTEM, n_issues=len(labeled_pri),
                         acc=round(acc,4), bal_acc=round(bal_acc,4),
                         mac_p=round(mac_p,4), mac_r=round(mac_r,4), mac_f1=round(mac_f1,4),
                         wgt_p=round(wgt_p,4), wgt_r=round(wgt_r,4), wgt_f1=round(wgt_f1,4)))
    print(f"Metrics saved → {path_pri_metrics}")

print(f"\nAll {len(issues_for_pri)} issues continue to bug assignment (no exclusion).")


[PRIORITY] Starting...

Running priority classification — model: claude-sonnet-4-6, system: A
Issues to process: 37


Priority | claude-sonnet-4-6:   0%|          | 0/37 [00:00<?, ?it/s]

Saved → /content/drive/MyDrive/triage_framework/results/priority_A_claude-sonnet-4-6.csv
[PRIORITY] Runtime: 108.0s (1.80 min)

  PRIORITY RESULTS (37 evaluated)
  Correct    : 21/37
  Accuracy   : 0.5676  |  Bal-Acc: 0.5020
  Mac  P/R/F1: 0.511 / 0.502 / 0.494
  Wgt  P/R/F1: 0.570 / 0.568 / 0.556
Metrics saved → /content/drive/MyDrive/triage_framework/results/metrics_priority_A_claude-sonnet-4-6.csv

All 37 issues continue to bug assignment (no exclusion).


## Task 4 — Bug Assignment

Best model: `gpt-5.5` System A  
Evaluated on 37 labeled issues

In [ ]:
assign_start = time.time()
print(f'[BUG ASSIGN] Starting...')

# ============================================================
# CELL 11 — Bug Assignment
# Best model: gpt-5.5 System A
# ============================================================
from tqdm.notebook import tqdm
import csv, json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

ASSIGN_MODEL  = 'gpt-5.5'           # ← best model (active) — Acc=0.973, Mac-F1=0.977
# ASSIGN_MODEL = 'gpt-4o-mini'
# ASSIGN_MODEL = 'gpt-4o'
# ASSIGN_MODEL = 'gpt-4.1-mini'
# ASSIGN_MODEL = 'gpt-4.1'
# ASSIGN_MODEL = 'gpt-5.4-mini'
# ASSIGN_MODEL = 'gpt-5.4'
# ASSIGN_MODEL = 'claude-haiku-4-5'
# ASSIGN_MODEL = 'claude-sonnet-4-6'
# ASSIGN_MODEL = 'claude-opus-4-7'

ASSIGN_SYSTEM = 'A'   # ← best system (active)
# ASSIGN_SYSTEM = 'B'
RAG_K         = 5

PACKAGE_HINTS = """
IMPORTANT — Historical assignment patterns (use these as strong prior signals).
Read ALL sections carefully before predicting — many errors come from applying
a pattern too broadly without checking the specific sub-component or package.

─── MdeModulePkg / HII / HiiDatabase / SetupBrowser / UI forms ───────────────
  → YangGangUEFI has been assigned: HiiDatabaseDxe, GetNameElement, SetupBrowserDxe bugs
  → EricGao2015 has been assigned: Device Manager forms, Setup Hotkey display, Assert/string-id bugs
  → lgao4 has been assigned: core DXE infrastructure bugs (InstallProtocolInterface, protocol/handle
    management, general low-level MdeModulePkg issues NOT covered by YangGangUEFI or EricGao2015)
  → wenbhou has been assigned: variable store / FTW (Fault Tolerant Write) spare block recovery bugs
  WARNING: Do NOT assign variable store or FTW bugs to os-d — os-d handles physical memory,
  not variable storage. Do NOT assign InstallProtocolInterface bugs to YangGangUEFI — that is
  a core DXE function belonging to lgao4.

─── mikebeaton — IMPORTANT: CLANGPDB scope is LIMITED to RedfishPkg only ─────
  → mikebeaton has been assigned: CLANGPDB build/link errors ONLY in RedfishPkg
  → Do NOT assign a CLANGPDB bug to mikebeaton if the package is NOT RedfishPkg
  → If title says "CLANGPDB" but package is NOT RedfishPkg → check other assignees

─── CLANGPDB linker bugs OUTSIDE RedfishPkg ──────────────────────────────────
  → deeglaze has been assigned: CLANGPDB linker warnings in non-RedfishPkg contexts
    (e.g. "/align without /driver" warnings in general build toolchain)
  WARNING: "CLANGPDB" alone does NOT mean mikebeaton. Always check the package first.

─── NetworkPkg ───────────────────────────────────────────────────────────────
  → BritChesley has been assigned: CodeQL / static-analysis bugs in NetworkPkg
  → SaloniKasbekar has been assigned: hardcoded-value / define-constant bugs in NetworkPkg
  → AndreasBBS has been assigned: QEMU_EFI + NetworkPkg combined driver-loading bugs
    (bugs where BOTH OvmfPkg/QEMU_EFI AND NetworkPkg are involved together)
  WARNING: If the bug is NetworkPkg + QEMU_EFI combined → AndreasBBS, NOT SaloniKasbekar/BritChesley

─── OvmfPkg / memory management — os-d vs vit9696 are DIFFERENT sub-domains ──
  → os-d has been assigned: PHYSICAL memory bugs — FreePages(), page table inconsistency,
    GCD data structures, memory-protection feature bugs, MdePkg LNK2001 linker errors
  → vit9696 has been assigned: POLICY bugs — runtime driver executability after EndOfDxe,
    driver loading permissions, UEFI runtime driver access control
  CRITICAL: A bug about "drivers not executable" or "EndOfDxe" → vit9696, NOT os-d
  CRITICAL: MdePkg LNK2001 unresolved symbol errors (e.g. __security_pop_cookie) → os-d,
    NOT ardbiesheuvel (ardbiesheuvel handles CryptoPkg VS2022 build errors, not MdePkg linker)

─── OvmfPkg / RISC-V ────────────────────────────────────────────────────────
  → yechao-w has been assigned: QEMU-KVM guest-boot failure bugs on RISC-V
  → zevorn has been assigned: OS-runtime crash bugs in RISC-V timer libs (BaseRiscV64CpuTimerLib)
  → jxu023 has been assigned: AMD SEV / SEV-ES virtualisation boot bugs in OvmfPkg
    (bugs mentioning AmdSvsm, SEV, SEV-ES → jxu023, NOT yechao-w)
  WARNING: yechao-w is RISC-V QEMU boot only. SEV/AMD bugs → jxu023.

─── OvmfPkg / Arm platform firmware menus ────────────────────────────────────
  → mikebeaton has been assigned: bugs where a specific commit broke key response
    in Arm platform firmware menus (regression bugs referencing a commit hash, NOT general UI)
  WARNING: Do NOT assign Arm firmware menu key-response regression bugs to YangGangUEFI or
  EricGao2015 — these are firmware-level regressions handled by mikebeaton, not HII/UI bugs.

─── ShellPkg / SMBIOS / SmbioView ───────────────────────────────────────────
  → praveensankarn has been assigned: ARMV9 / processor support additions in SmbioView
  → NingFengGit has been assigned: MRDIMM / MemoryDevice table additions in ShellPkg SMBIOS

─── CryptoPkg / build errors ─────────────────────────────────────────────────
  → mdkinney has been assigned: CLANG compiler build error bugs in CryptoPkg
  → ardbiesheuvel has been assigned: VS2022 build error bugs in CryptoPkg
  → liyi77 has been assigned: OpenSSL version update / CVE patch bugs in CryptoPkg
  WARNING: ardbiesheuvel is CryptoPkg + VS2022 ONLY. MdePkg linker errors → os-d, not ardbiesheuvel.

─── BaseTools / CI ───────────────────────────────────────────────────────────
  → bexcran has been assigned: ECC Check / CI failure bugs in BaseTools
  → makubacki has been assigned: NASM / toolchain / nuget update bugs in BaseTools

─── Assignees with unique bug types ─────────────────────────────────────────
  → HunterChang030: SecurityPkg / OpalPassword device list handling bugs
  → philnoh2: UefiCpuPkg BSP/AP multiprocessor synchronisation bugs
  → praravin: IntelFsp2Pkg / FSP-T TemporaryRamSize configuration bugs
  When a bug clearly matches one of these descriptions, use that assignee even at low
  confidence — do NOT default to a more frequently seen assignee.
"""
_ASSIGN_TEMPLATE = """You are a bug triage assistant for the tianocore/edk2 UEFI firmware repository.

{package_hints}
---
Below are past bug issues and their assignees:

{examples}

---
New bug to assign:

{test_issue}

Before predicting, reason through these steps:
  1. What is the exact package (e.g. RedfishPkg, MdePkg, SecurityPkg)?
  2. What is the specific sub-component or bug type within that package?
  3. Which historical pattern matches most precisely?
  4. Is there a rare assignee listed in the hints who matches better than a common one?

Only choose from this list: {known}

Respond ONLY with a JSON object, nothing else:
{{"predicted_assignee": "<login>", "confidence": <float 0.0-1.0>, "reason": "<one sentence>"}}"""

def fmt_issue_example(issue, include_assignee=True):
    useful_labels = []
    if issue.get('package') and issue['package'] != '-':
        useful_labels.append(f"package:{issue['package']}")
    if issue.get('priority') and issue['priority'] != '-':
        useful_labels.append(f"priority:{issue['priority']}")
    label_str    = ', '.join(useful_labels) if useful_labels else 'none'
    body_preview = str(issue['title'])[:300].replace('\n', ' ').strip()
    text = (f"Title  : {issue['title']}\n"
            f"Labels : {label_str}\n"
            f"Body   : {body_preview}")
    if include_assignee and issue.get('true_assignee'):
        text += f"\nAssignee: {issue['true_assignee']}"
    return text

def fmt_issue_query(issue):
    return fmt_issue_example(issue, include_assignee=False)

def parse_assignment(raw_result):
    if raw_result is None: return None
    assignee = raw_result.get('predicted_assignee','').strip()
    if assignee not in KNOWN_ASSIGNEES:
        match    = next((a for a in KNOWN_ASSIGNEES if a.lower()==assignee.lower()), None)
        assignee = match if match else KNOWN_ASSIGNEES[0]
        raw_result['predicted_assignee'] = assignee
        raw_result['corrected']          = True
    return raw_result

def predict_assign_A(issue, model=ASSIGN_MODEL):
    """System A: Leave-one-out — all 37 labeled issues except current one."""
    examples = [i for i in labeled_issues if i['id'] != issue['id']]
    prompt   = _ASSIGN_TEMPLATE.format(
        package_hints = PACKAGE_HINTS,
        examples      = '\n\n'.join(fmt_issue_example(i) for i in examples),
        test_issue    = fmt_issue_query(issue),
        known         = ', '.join(sorted(KNOWN_ASSIGNEES)),
    )
    return parse_assignment(call_llm(prompt, ' ', model, max_tokens=500))

def predict_assign_B(issue, model=ASSIGN_MODEL):
    """System B: RAG — top-k similar labeled issues."""
    similar  = get_similar_labeled(issue, k=RAG_K)
    prompt   = _ASSIGN_TEMPLATE.format(
        package_hints = PACKAGE_HINTS,
        examples      = '\n\n'.join(fmt_issue_example(i) for i in similar),
        test_issue    = fmt_issue_query(issue),
        known         = ', '.join(sorted(KNOWN_ASSIGNEES)),
    )
    return parse_assignment(call_llm(prompt, ' ', model, max_tokens=500))

predict_assign_fn = predict_assign_A if ASSIGN_SYSTEM == 'A' else predict_assign_B

# ── Run bug assignment ────────────────────────────────────────
# Always evaluate on the 37 labeled issues regardless of pipeline
print(f"\nRunning bug assignment — model: {ASSIGN_MODEL}, system: {ASSIGN_SYSTEM}")
print(f"Evaluating on all {len(labeled_issues)} labeled issues (no exclusion from previous tasks)")

assign_results  = []
path_assign_csv = f'{RESULTS_DIR}/assignee_{ASSIGN_SYSTEM}_{ASSIGN_MODEL.replace(".","_")}.csv'

for issue in tqdm(labeled_issues, desc=f'Assign | {ASSIGN_MODEL}'):
    try:
        pred = predict_assign_fn(issue)
        if pred is None:
            pred = {'predicted_assignee': None, 'confidence': 'low', 'reason': 'error'}
        assign_results.append({
            'model'              : ASSIGN_MODEL,
            'system'             : ASSIGN_SYSTEM,
            'issue_id'           : issue['id'],
            'title'              : issue['title'],
            'package'            : issue['package'],
            'type'               : issue.get('type','-'),
            'priority'           : issue.get('priority','-'),
            'predicted_assignee' : pred.get('predicted_assignee'),
            'confidence'         : pred.get('confidence','low'),
            'reason'             : pred.get('reason',''),
            'true_assignee'      : issue['true_assignee'],
            'correct'            : pred.get('predicted_assignee') == issue['true_assignee'],
        })
    except Exception as e:
        print(f"  Skipped #{issue['id']}: {e}")

# Save CSV
keys = ['model','system','issue_id','title','package','type','priority',
        'predicted_assignee','confidence','reason','true_assignee','correct']
with open(path_assign_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
    writer.writeheader(); writer.writerows(assign_results)

# ── Evaluation ────────────────────────────────────────────────
labeled_r = [r for r in assign_results if r['true_assignee'] is not None and r['predicted_assignee'] is not None]
if labeled_r:
    y_true  = [r['true_assignee']      for r in labeled_r]
    y_pred  = [r['predicted_assignee'] for r in labeled_r]
    correct = sum(1 for r in labeled_r if r['correct'])
    acc     = accuracy_score(y_true, y_pred)
    mac_p   = precision_score(y_true, y_pred, average='macro',    zero_division=0)
    mac_r   = recall_score(   y_true, y_pred, average='macro',    zero_division=0)
    mac_f1  = f1_score(       y_true, y_pred, average='macro',    zero_division=0)
    wgt_p   = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    wgt_r   = recall_score(   y_true, y_pred, average='weighted', zero_division=0)
    wgt_f1  = f1_score(       y_true, y_pred, average='weighted', zero_division=0)
    print(f"\n{'='*55}")
    print(f"  BUG ASSIGNMENT RESULTS ({len(labeled_r)} evaluated)")
    print(f"{'='*55}")
    print(f"  Correct  : {correct}/{len(labeled_r)}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Mac  P/R/F1: {mac_p:.3f} / {mac_r:.3f} / {mac_f1:.3f}")
    print(f"  Wgt  P/R/F1: {wgt_p:.3f} / {wgt_r:.3f} / {wgt_f1:.3f}")
    print(f"{'='*55}")
    print(f"Saved → {path_assign_csv}")

# ── Save evaluation metrics to CSV ───────────────────────────
if labeled_r:
    path_assign_metrics = f'{RESULTS_DIR}/metrics_assignee_{ASSIGN_SYSTEM}_{ASSIGN_MODEL.replace(".","_")}.csv'
    with open(path_assign_metrics, 'w', newline='') as _f:
        _w = csv.DictWriter(_f, fieldnames=['model','system','n_issues','correct','acc',
                                             'mac_p','mac_r','mac_f1',
                                             'wgt_p','wgt_r','wgt_f1'])
        _w.writeheader()
        _w.writerow(dict(model=ASSIGN_MODEL, system=ASSIGN_SYSTEM, n_issues=len(labeled_r),
                         correct=correct, acc=round(acc,4),
                         mac_p=round(mac_p,4), mac_r=round(mac_r,4), mac_f1=round(mac_f1,4),
                         wgt_p=round(wgt_p,4), wgt_r=round(wgt_r,4), wgt_f1=round(wgt_f1,4)))
    print(f"Metrics saved → {path_assign_metrics}")
assign_elapsed = time.time() - assign_start
print(f'[BUG ASSIGN] Runtime: {assign_elapsed:.1f}s ({assign_elapsed/60:.2f} min)')


[BUG ASSIGN] Starting...

Running bug assignment — model: gpt-5.5, system: A
Evaluating on all 37 labeled issues (no exclusion from previous tasks)


Assign | gpt-5.5:   0%|          | 0/37 [00:00<?, ?it/s]


  BUG ASSIGNMENT RESULTS (37 evaluated)
  Correct  : 36/37
  Accuracy : 0.9730
  Mac  P/R/F1: 0.983 / 0.983 / 0.977
  Wgt  P/R/F1: 0.986 / 0.973 / 0.973
Saved → /content/drive/MyDrive/triage_framework/results/assignee_A_gpt-5_5.csv
Metrics saved → /content/drive/MyDrive/triage_framework/results/metrics_assignee_A_gpt-5_5.csv
[BUG ASSIGN] Runtime: 154.4s (2.57 min)


In [ ]:
# ============================================================
# CELL 12b — Merge all predictions into one unified CSV
# ============================================================
import pandas as pd
import os

print("Individual CSVs saved per task:")
print(f"  Invalid    : {path_inv_csv}")
print(f"  Duplicate  : {path_dup_csv}")
print(f"  Priority   : {path_pri_csv}")
print(f"  Assignment : {path_assign_csv}")

# ── Build unified per-issue summary ──────────────────────────
# One row per issue with predictions from all 4 tasks
issue_map = {i['id']: {'issue_id': i['id'], 'title': i['title'],
                        'package': i['package'], 'true_priority': i.get('priority','-'),
                        'true_assignee': i.get('true_assignee','')} for i in labeled_issues}

# Add invalid predictions
for r in invalid_results:
    iid = r['issue_id']
    if iid in issue_map:
        issue_map[iid]['pred_invalid']    = r.get('is_invalid', False)
        issue_map[iid]['invalid_conf']    = r.get('confidence', '')
        issue_map[iid]['invalid_reason']  = r.get('reason', '')

# Add duplicate predictions
for r in dup_results:
    iid = r['issue_id']
    if iid in issue_map:
        issue_map[iid]['pred_duplicate']       = r.get('is_duplicate', False)
        issue_map[iid]['duplicate_of']         = r.get('duplicate_of', '')
        issue_map[iid]['duplicate_title']      = r.get('duplicate_title', '')
        issue_map[iid]['duplicate_conf']       = r.get('confidence', '')

# Add priority predictions
for r in pri_results:
    iid = r['issue_id']
    if iid in issue_map:
        issue_map[iid]['pred_priority']   = r.get('predicted_priority', '')
        issue_map[iid]['priority_conf']   = r.get('confidence', '')
        issue_map[iid]['priority_reason'] = r.get('reason', '')
        issue_map[iid]['priority_correct'] = (
            str(r.get('predicted_priority','')).lower() ==
            str(r.get('true_priority','')).lower()
        )

# Add assignment predictions
for r in assign_results:
    iid = r['issue_id']
    if iid in issue_map:
        issue_map[iid]['pred_assignee']    = r.get('predicted_assignee', '')
        issue_map[iid]['assignee_conf']    = r.get('confidence', '')
        issue_map[iid]['assignee_reason']  = r.get('reason', '')
        issue_map[iid]['assign_correct']   = r.get('correct', False)

# Build DataFrame
unified_cols = [
    'issue_id', 'title', 'package',
    # Invalid
    'pred_invalid', 'invalid_conf', 'invalid_reason',
    # Duplicate
    'pred_duplicate', 'duplicate_of', 'duplicate_title', 'duplicate_conf',
    # Priority
    'true_priority', 'pred_priority', 'priority_conf', 'priority_reason', 'priority_correct',
    # Assignment
    'true_assignee', 'pred_assignee', 'assignee_conf', 'assignee_reason', 'assign_correct',
]

rows = []
for iid in sorted(issue_map.keys(), key=int):
    row = issue_map[iid]
    rows.append({col: row.get(col, '') for col in unified_cols})

df_unified = pd.DataFrame(rows, columns=unified_cols)

# Cast boolean columns to proper bool type
for _bool_col in ['pred_invalid', 'pred_duplicate', 'priority_correct', 'assign_correct']:
    if _bool_col in df_unified.columns:
        df_unified[_bool_col] = df_unified[_bool_col].map(
            lambda x: True if str(x).strip().lower() in ('true','1','yes') else False
        )

path_unified = f'{RESULTS_DIR}/unified_predictions_37issues.csv'
df_unified.to_csv(path_unified, index=False)
print(f"\n✅ Unified predictions CSV saved → {path_unified}")
print(f"   Rows: {len(df_unified)}  |  Columns: {len(df_unified.columns)}")
print(f"\nColumn overview:")
for c in unified_cols:
    print(f"  {c}")

# Quick summary
print(f"\n{'='*55}")
print(f"  PIPELINE SUMMARY — 37 labeled issues")
print(f"{'='*55}")
if 'pred_invalid' in df_unified.columns:
    print(f"  Flagged invalid   : {df_unified['pred_invalid'].sum()}")
if 'pred_duplicate' in df_unified.columns:
    print(f"  Flagged duplicate : {df_unified['pred_duplicate'].sum()}")
if 'priority_correct' in df_unified.columns:
    pc = df_unified['priority_correct'].sum()
    print(f"  Priority correct  : {pc}/{len(df_unified)}  ({pc/len(df_unified)*100:.1f}%)")
if 'assign_correct' in df_unified.columns:
    ac = df_unified['assign_correct'].sum()
    print(f"  Assignment correct: {ac}/{len(df_unified)}  ({ac/len(df_unified)*100:.1f}%)")
print(f"{'='*55}")


Individual CSVs saved per task:
  Invalid    : /content/drive/MyDrive/triage_framework/results/invalid_A_claude-sonnet-4-6.csv
  Duplicate  : /content/drive/MyDrive/triage_framework/results/dup_A_claude-sonnet-4-6.csv
  Priority   : /content/drive/MyDrive/triage_framework/results/priority_A_claude-sonnet-4-6.csv
  Assignment : /content/drive/MyDrive/triage_framework/results/assignee_A_gpt-5_5.csv

✅ Unified predictions CSV saved → /content/drive/MyDrive/triage_framework/results/unified_predictions_37issues.csv
   Rows: 37  |  Columns: 20

Column overview:
  issue_id
  title
  package
  pred_invalid
  invalid_conf
  invalid_reason
  pred_duplicate
  duplicate_of
  duplicate_title
  duplicate_conf
  true_priority
  pred_priority
  priority_conf
  priority_reason
  priority_correct
  true_assignee
  pred_assignee
  assignee_conf
  assignee_reason
  assign_correct

  PIPELINE SUMMARY — 37 labeled issues
  Flagged invalid   : 1
  Flagged duplicate : 0
  Priority correct  : 21/37  (56.8%)
  

## Summary — Integrated Pipeline Results

In [ ]:
# ============================================================
# CELL 12 — Integrated Pipeline Summary
# ============================================================
import pandas as pd

print("=" * 65)
print("  INTEGRATED TRIAGE FRAMEWORK — PIPELINE SUMMARY")
print("=" * 65)

# Invalid
if 'invalid_results' in dir() and invalid_results:
    n_inv = sum(1 for r in invalid_results if r['is_invalid'])
    print(f"\n[1] INVALID DETECTION  (model: {INVALID_MODEL}, sys: {INVALID_SYSTEM})")
    print(f"    Issues processed : {len(invalid_results)}")
    print(f"    Flagged invalid  : {n_inv}")
    print(f"    Passed forward   : {len(invalid_results) - n_inv}")

# Duplicate
if 'dup_results' in dir() and dup_results:
    n_dup = sum(1 for r in dup_results if r['is_duplicate'])
    print(f"\n[2] DUPLICATE DETECTION (model: {DUP_MODEL}, sys: {DUP_SYSTEM})")
    print(f"    Issues processed : {len(dup_results)}")
    print(f"    Flagged duplicate: {n_dup}")
    print(f"    Passed forward   : {len(dup_results) - n_dup}")
    if n_dup > 0:
        flagged = [(r['issue_id'], r['title'][:50], r['duplicate_of'])
                   for r in dup_results if r['is_duplicate']]
        for iid, title, dup_of in flagged:
            print(f"    ↳ #{iid} '{title}' → dup of #{dup_of}")

# Priority
if 'pri_results' in dir() and pri_results:
    from collections import Counter
    dist = Counter(r.get('predicted_priority','?') for r in pri_results)
    print(f"\n[3] PRIORITY CLASSIFICATION (model: {PRI_MODEL}, sys: {PRI_SYSTEM})")
    print(f"    Issues processed : {len(pri_results)}")
    for label in ['high','medium','low']:
        print(f"    {label:8s}: {dist.get(label,0)}")

# Bug assignment
if 'assign_results' in dir() and assign_results:
    from collections import Counter
    assignee_dist = Counter(r['predicted_assignee'] for r in assign_results if r.get('predicted_assignee'))
    print(f"\n[4] BUG ASSIGNMENT  (model: {ASSIGN_MODEL}, sys: {ASSIGN_SYSTEM})")
    print(f"    Issues evaluated : {len(assign_results)}")
    correct_count = sum(1 for r in assign_results if r.get('correct'))
    print(f"    Correct          : {correct_count}/{len(assign_results)}")
    print(f"    Assignee distribution:")
    for assignee, count in sorted(assignee_dist.items(), key=lambda x:-x[1]):
        print(f"      {assignee:20s}: {count}")

print("\n" + "=" * 65)
print(f"  All results saved to: {RESULTS_DIR}")
print("=" * 65)


# ── Runtime summary ───────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  RUNTIME SUMMARY")
print(f"{'='*65}")
task_times = {
    'Invalid Detection'      : locals().get('invalid_elapsed', None),
    'Duplicate Detection'    : locals().get('dup_elapsed',     None),
    'Priority Classification': locals().get('pri_elapsed',     None),
    'Bug Assignment'         : locals().get('assign_elapsed',  None),
}
for task, t in task_times.items():
    if t is not None:
        print(f"  {task:<25}: {t:.1f}s  ({t/60:.2f} min)")
    else:
        print(f"  {task:<25}: not run")

if 'setup_elapsed' in dir():
    print(f"  {'─'*40}")
    print(f"  {'Setup (CSV+index)':<25}: {setup_elapsed:.1f}s  ({setup_elapsed/60:.2f} min)")

inference_elapsed = time.time() - inference_start_time if 'inference_start_time' in dir() else None
if inference_elapsed is not None:
    print(f"  {'Inference (LLM only)':<25}: {inference_elapsed:.1f}s  ({inference_elapsed/60:.2f} min)")

if 'pipeline_start_time' in dir():
    total_elapsed = time.time() - pipeline_start_time
    print(f"  {'─'*40}")
    print(f"  {'TOTAL (setup+inference)':<25}: {total_elapsed:.1f}s  ({total_elapsed/60:.2f} min)")
print(f"{'='*65}")


# ── Save runtime to CSV ───────────────────────────────────────
import csv as _csv
path_runtime_csv = f'{RESULTS_DIR}/runtime_summary.csv'
with open(path_runtime_csv, 'w', newline='') as _f:
    _w = _csv.DictWriter(_f, fieldnames=['task','model','system',
                                          'runtime_seconds','runtime_minutes'])
    _w.writeheader()
    task_info = [
        ('Invalid Detection',       INVALID_MODEL, INVALID_SYSTEM, locals().get('invalid_elapsed')),
        ('Duplicate Detection',     DUP_MODEL,     DUP_SYSTEM,     locals().get('dup_elapsed')),
        ('Priority Classification', PRI_MODEL,     PRI_SYSTEM,     locals().get('pri_elapsed')),
        ('Bug Assignment',          ASSIGN_MODEL,  ASSIGN_SYSTEM,  locals().get('assign_elapsed')),
    ]
    for task, model, system, t in task_info:
        if t is not None:
            _w.writerow(dict(task=task, model=model, system=system,
                             runtime_seconds=round(t,2),
                             runtime_minutes=round(t/60,4)))
    # Total
    if 'setup_elapsed' in dir():
        _w.writerow(dict(task='SETUP (CSV+index)', model='all', system='all',
                         runtime_seconds=round(setup_elapsed,2),
                         runtime_minutes=round(setup_elapsed/60,4)))
    if inference_elapsed is not None:
        _w.writerow(dict(task='INFERENCE (LLM only)', model='all', system='all',
                         runtime_seconds=round(inference_elapsed,2),
                         runtime_minutes=round(inference_elapsed/60,4)))
    if 'pipeline_start_time' in dir():
        total = time.time() - pipeline_start_time
        _w.writerow(dict(task='TOTAL (setup+inference)', model='all', system='all',
                         runtime_seconds=round(total,2),
                         runtime_minutes=round(total/60,4)))
print(f"Runtime saved → {path_runtime_csv}")


  INTEGRATED TRIAGE FRAMEWORK — PIPELINE SUMMARY

[1] INVALID DETECTION  (model: claude-sonnet-4-6, sys: A)
    Issues processed : 37
    Flagged invalid  : 1
    Passed forward   : 36

[2] DUPLICATE DETECTION (model: claude-sonnet-4-6, sys: A)
    Issues processed : 37
    Flagged duplicate: 0
    Passed forward   : 37

[3] PRIORITY CLASSIFICATION (model: claude-sonnet-4-6, sys: A)
    Issues processed : 37
    high    : 5
    medium  : 20
    low     : 12

[4] BUG ASSIGNMENT  (model: gpt-5.5, sys: A)
    Issues evaluated : 37
    Correct          : 36/37
    Assignee distribution:
      mikebeaton          : 3
      EricGao2015         : 3
      os-d                : 3
      YangGangUEFI        : 2
      mdkinney            : 2
      Damien-Chen         : 1
      yechao-w            : 1
      makubacki           : 1
      ardbiesheuvel       : 1
      zevorn              : 1
      bexcran             : 1
      NingFengGit         : 1
      philnoh2            : 1
      vit9696       